In [45]:
import pandas as pd

In [46]:
df = pd.read_csv("../data/raw/Faults.csv")

print("원본 데이터 크기:", df.shape)

원본 데이터 크기: (1941, 34)


In [47]:
# 중복 확인 및 처리
duplicate_count = df.duplicated().sum()
print("중복 행 수:", duplicate_count)

중복 행 수: 0


**확인 결과**: 완전히 동일한 중복 행은 0건으로 확인되어 별도의 중복 제거는 수행하지 않았다.

In [48]:
# Target 컬럼명 오타 수정
df = df.rename(columns={
    "K_Scatch": "K_Scratch"
})

## Target 단일 컬럼 생성

In [49]:
# Target 컬럼
target_cols = [
    "Pastry",
    "Z_Scratch",
    "K_Scratch",
    "Stains",
    "Dirtiness",
    "Bumps",
    "Other_Faults"
]

# 단일 multiclass target 생성
df["target"] = df[target_cols].idxmax(axis=1)

# 기존 one-hot target 제거
df_processed = df.drop(columns=target_cols)

print("전처리 전:", df.shape)
print("전처리 후:", df_processed.shape)

df_processed.head()

전처리 전: (1941, 35)
전처리 후: (1941, 28)


,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Edges_X_Index,Edges_Y_Index,Outside_Global_Index,LogOfAreas,Log_X_Index,Log_Y_Index,Orientation_Index,Luminosity_Index,SigmoidOfAreas,target
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.4706,1.0000,1.0,2.4265,0.9031,1.6435,0.8182,-0.2913,0.5822,Pastry
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.6000,0.9667,1.0,2.0334,0.7782,1.4624,0.7931,-0.1756,0.2984,Pastry
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.7500,0.9474,1.0,1.8513,0.7782,1.2553,0.6667,-0.1228,0.2150,Pastry
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.5385,1.0000,1.0,2.2455,0.8451,1.6532,0.8444,-0.1568,0.5212,Pastry
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.2833,0.9885,1.0,3.3818,1.2305,2.4099,0.9338,-0.1992,1.0000,Pastry


## 강종 변수 통합

`TypeOfSteel_A300`, `TypeOfSteel_A400`은 동일한 강종 정보를 상호 배타적으로 나타내는 이진 변수임을 확인하였다. 두 변수를 함께 사용할 경우 정보가 중복되므로 `TypeOfSteel_A300`만 유지하였다. 1은 A300, 0은 A400을 의미한다.

In [50]:
df_processed[
    ["TypeOfSteel_A300", "TypeOfSteel_A400"]
].value_counts().sort_index()

TypeOfSteel_A300  TypeOfSteel_A400
0                 1                   1164
1                 0                    777
Name: count, dtype: int64

In [51]:
# 강종 변수 정리
# A300과 A400은 상호 배타적인 이진 변수이므로 A300 여부만 유지
df_processed = df_processed.drop(columns=["TypeOfSteel_A400"])

In [52]:
print(df_processed["TypeOfSteel_A300"].value_counts())
print("최종 데이터 크기:", df_processed.shape)

TypeOfSteel_A300
0    1164
1     777
Name: count, dtype: int64
최종 데이터 크기: (1941, 27)


In [53]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "Faults_processed.csv"

df_processed.to_csv(output_path, index=False)

print(f"저장 완료: {output_path}")
print(f"데이터 크기: {df_processed.shape}")

저장 완료: ..\data\processed\Faults_processed.csv
데이터 크기: (1941, 27)


In [54]:
check_df = pd.read_csv("../data/processed/Faults_processed.csv")

print("저장된 데이터 크기:", check_df.shape)
print("중복 행:", check_df.duplicated().sum())
print("결측치 수:", check_df.isnull().sum().sum())
print("\nTarget 분포:")
print(check_df["target"].value_counts())

check_df.head()

저장된 데이터 크기: (1941, 27)
중복 행: 0
결측치 수: 0

Target 분포:
target
Other_Faults    673
Bumps           402
K_Scratch       391
Z_Scratch       190
Pastry          158
Stains           72
Dirtiness        55
Name: count, dtype: int64


,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Edges_X_Index,Edges_Y_Index,Outside_Global_Index,LogOfAreas,Log_X_Index,Log_Y_Index,Orientation_Index,Luminosity_Index,SigmoidOfAreas,target
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.4706,1.0000,1.0,2.4265,0.9031,1.6435,0.8182,-0.2913,0.5822,Pastry
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.6000,0.9667,1.0,2.0334,0.7782,1.4624,0.7931,-0.1756,0.2984,Pastry
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.7500,0.9474,1.0,1.8513,0.7782,1.2553,0.6667,-0.1228,0.2150,Pastry
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.5385,1.0000,1.0,2.2455,0.8451,1.6532,0.8444,-0.1568,0.5212,Pastry
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.2833,0.9885,1.0,3.3818,1.2305,2.4099,0.9338,-0.1992,1.0000,Pastry


In [55]:
train_idx = pd.read_csv("../data/split/train_idx.csv")["index"]
test_idx = pd.read_csv("../data/split/test_idx.csv")["index"]

print("전체 데이터:", len(df_processed))
print("Train:", len(train_idx))
print("Test:", len(test_idx))
print("Train + Test:", len(train_idx) + len(test_idx))

print("Train/Test 중복:", len(set(train_idx) & set(test_idx)))
print(
    "전체 행 포함:",
    set(train_idx) | set(test_idx) == set(range(len(df_processed)))
)

전체 데이터: 1941
Train: 1552
Test: 389
Train + Test: 1941
Train/Test 중복: 0
전체 행 포함: True
